# ConvNext



> If CNNs borrowed the best ideas from Vision Transformers, could CNNs become competitive again ?

`Answer`:  Yes, They start with ResNet-50 and modernize it step by step.

Image model is a machine that repeatedly does two things:
1. Mix spatial information: "what is near that"?
2. Mix channel information: "what features should i combine?"

ConvNeXt changes how efficinetly and effectively does these.

1. Better training - teach the same model better
Original ResNet training : 90 epochs -> 76.1%

while,

ConvNeXt first gives a ResNet a modern Transformer-style training:

- 300 epochs -> more time to learn
- AdamW -> better controlled weight decay than Adam
- Mixup -> blend two images
- RandAugment -> random transformers
- Random Erasing -> erase random regions
- Stochastic depth + Label Smoothing -> reduce Overfitting

Result: from 76.1% to 78.8%


> Before changing the machine, make sure we are teaching it well.


2. Macro Design - Chage the overall Shape

- Stage ratio:
ResNet has 4 stages 3:4:6:3 blocks.

Swin transformer uses roughly:
1:1:3:1

so ConvNext changes ResNet to: 3:3:9:3

This puts more computation in the deeper middle stages, which made 78.8 → 79.4%.

-  Replace the stem with patchify:

Old ResNet: $7*7$ conv, stride 2 -> max pool

new: $4*4$ conv, stride 4

Just like, Cutting the image into non-overlapping $4*4$ patches and processing them. Both reduce the image resolution 4*, but patchifying and resembles ViTs.

79.4 → 79.5%

ConvNext starts looking more like a transformer at the network level structure.


- ResNeXt-ify - Separate spatial and channel mixing
Normal convolution mixes everything together.

ConvNext uses:

* Depthwise convolution -> spatial mixing
* $1*1$ convolution -> channel mixing

A depthwise convolution applies one spatial filter per channel so it is much cheaper.

but cheaper can mean less expressive.

So ConvNeXt increases the channels: 64 -> 96

This recovers performance without greatly increasing FLOPs.

Let one operation answe "where?" and another answer "what feature" similar to how transformers seperate spatial/token mixing from feature transformation.



- Inverted Bottleneck - expand before processing.

A normal bottleneck often does wide → narrow → wide

transformers style blocks instead do narrow → VERY wide → narrow.

ConvNeXt: 96 → 384 → 96

Because the wider hidden representation gives the network more room to manipulate information. This idea comes `MobileNetV2` and is common from transformers.

Temporarily gives the network a much bigger workspace


- Large Kernels - see a bigger neighborhood

A $3*3$ convolution sees a small neighbourhood. transformers can capture relationships over much larger regions. Swin uses windows around $7*7$.

so  ConvNext changes: $3*3$ -> $7*7$ depthwise convolution.
but first, they move the depthwise convolution earlier in the block.

Because:
- Depthwise 7*7 = cheaply gather spatial information from a large area.
- 1*1 convolution =efficiently mix channels.

This gives CNNs a larger receptive field - the amount of sorrounding image formation a unit can see.

A bigger window lets the CNN see more of the image,  somewhat like attentions broader context.


- Micro design -Fix the small details: Now ConvNext cleans up the individual block.

*Activation*: ReLu -> GELU

And instead of many activations, ConvNext keeps essentially one GELU between the two 1*1 layers.

*Normalization* Instead of several batchNorm layers:

BatchNorm -> LayerNorm

and fewer normalization operations overall.

*Downsampling* Instead of hiding downsampling inside other operations, ConvNext adds a seperate downsampling layer between stages.

These seemingly small changes matter.

80.6 → 82.0%

ConvNeXt ends up beating the reported 81.3% Swin Transformer result.






> CNN = a neural network that uses convolution filters to progressively turn pixels into meaningful visual features.

> Note: Swin Transformer is a Vision Transformer(viT) designed to work efficiently on images. A normal Transformer looks at relationships between tokens. Swin does the same thing, but it looks at small local windows and gradually combines them into larger representations.

> ConvNeXt is basically a modernized ResNet that borrows the training, structure, normalization, expansion, and large-context ideas that made Vision Transformers powerful—while keeping convolution at its core. ConvNeXt shows that you don't need attention to get Transformer-like performance—you can make a CNN much stronger by adopting the design principles that made Vision Transformers successful.




# Transfer learning

> Take a model that has been already trained and learned something useful and reuse that knowledge for a new task. Instead of starting from zero and train everything, use pretrained model, reuse what it knows and adapt it to task.

but,

**Fine tuning**: Start with pre-trained model but continue training some of its weight on new dataset.


## Whats get reused ?

A vison model is:

- Feature extraction -> Feature procesing -> Final task

For example; Image -> edges/texture/shapes -> high level features -> dog breed

The early layers learn a general features.
so usually:
  - Early layers: keep them frozen -> know general visual concepts
  - Later layers: fine tune them, to convey need to adapt specific task.
  - Final layers: Often replace/retrain it because task may have different classes.

We can keep most of the feature knowledge and change the final classifier.

**Why is this useful?**
<br>
Because training from scratch requires:
  - Lots of labeled data
  - Lots of computation
  - lots of time
A pretrained model has alreadt done much of expensive learning.
So transfer learning is faster, cheaper and require less labeled data.


Transfer learning might not always help, the old knowledge needs to be relevant, it can hurt performance which is related to negative transfer.


```
Transfer learning = reuse pretrained knowledge.
Fine-tuning = adjust that pretrained knowledge for your new task.
Early layers = general knowledge. Later layers = task-specific knowledge.
Main benefit = less data, time, and compute than training from scratch.
```


> A Transformer is a neural network that repeatedly lets each piece of a sequence look at the other pieces, figure out which ones matter, transform that information, and use the resulting context to make predictions. Instead of reading a sentence strictly one word at a time, the model can look at many words and learn how they relate to one another.

## MobileNet + Transformers

MobileNet is lightweight CNN; Transformers are good at understanding relationships between different parts of an image. What if we combine them ?


MobileNet uses convolutions, especially depthwise seperable convolutions, to efficiently extract local features.

pixel -> edges -> textures -> shapes

designed to be fast and small for phones and edge devices.


Transformers :  A vision transformer uses attention to understand relationships between different image regions.
> This part of image is related to that part.

so,
- MobileNet : efficiently local feature extraction
- Transformer: globa/ contextual understanding


**Mobile-Formers**

<br>

```
It combines both ideas into one architecture.

Image → MobileNet → local visual information
↕
Transformer → global/context information
```

two components interact, allowing the model to get the efficiency of MobileNet while benefitting from Transformer-style attention.

so the idea is "can we combine lightweight CNN and a transformer so each does what it is good at ?"

Thats what Mobile-Former explores.

```
MobileNet = the model
Mobile-Former = CNN + Transformer architecture
timm = a library that gives you easy access to pretrained vision models
```

*timm* : Pytorch image models, simply a python library containing many computer vision models, including pretrained MobileNet Variants.

In [ ]:
!pip install timm

In [ ]:
import timm
import torch

model_name="mobilenetv3_large_100"

model=timm.create_model(model_name,pretrained=True)

model.eval()  # to use model for inference

# forward pass with dummy input, batch size 1,3 and color channels 224*224 image
input_tensor=torch.rand(1,3,224,224)
output=model(input_tensor)
print(output)

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.1MB            

model.safetensors: downloading bytes:           |  0.00B            

tensor([[-2.6443e+00, -6.2406e-02,  1.9040e-01, -1.8231e-01,  1.1388e+00,
          2.5458e-01,  8.4652e-01, -3.2164e-01, -2.3258e-01, -5.1753e-01,
         -4.1408e-01,  6.3645e-02,  3.0202e-01,  2.1535e-01,  2.6603e-02,
          6.0678e-01,  1.1158e+00, -3.8084e-01,  1.6429e+00,  8.9635e-01,
          4.5469e-01,  2.9433e+00,  1.6785e+00,  2.4056e+00,  5.5167e-01,
         -1.2561e-01, -1.6196e+00, -4.8791e-01, -9.3673e-01, -1.6519e+00,
         -1.1569e+00, -8.9671e-01, -1.7348e+00,  1.4082e+00,  4.2182e+00,
         -1.3283e+00, -1.3510e+00, -1.3030e+00,  1.7596e-03, -8.6705e-01,
         -4.6259e-01, -1.5491e-01,  1.6073e+00,  3.5735e-01, -2.2777e-01,
          3.3875e-01, -1.5900e-03, -4.0284e-01, -6.2338e-01, -1.7685e+00,
         -2.0033e-01, -9.4366e-02,  6.1136e-02,  3.0804e-01,  1.3545e+00,
         -1.6688e-02,  5.7412e-01, -1.0981e+00,  1.5115e+00, -7.7753e-01,
          2.0292e+00,  1.9133e-01,  6.1269e-02,  1.5418e-01, -2.3428e-01,
          3.7037e+00,  2.0216e+00,  1.

# ResNet / Residual Block

ResNet makes very deep neural network easier to train by adding shortcuts that lets information and gradient skip layers.

1. The normal problem
A normal block tries to learn:
    * x-> F(x) ->y
meaning:
"take the input x and completely transform it into desired output" with many layers, this becomes difficult to optimize.

2. ResNet`s trick:
x -> F(x)

x ─────────→ + → y

so: y = F(x) + x

The x is going directly around the layers is the skip/shortcut connection.

This is easier because let the best thing to do is basically leave x unchanged.
A normal network has to learn:

"copy x perfectly"

A ResNet can simply make: F(x) ≈ 0 then y=0+x=x much easier.

3. "residual":
The network is effectively learning:
R(x)=desired output - x

so instead of learning the entire transformation. it learns the change/residuals that needs to be added to x.

"dont learn the whole answer, learn what needs to be changed2

4. Why it helps gradients ?
During backpropagation:

y = F(x) + x

The gradients has two paths:
1. Through F(x)
2. Directly through the shortcut
The shortcut gives the gradient a relatively clean path backward. so even if the layers inside F(x) have difficulty passing gradients, the shortcut helps gradients reach earlier layers.


Therefore:
> Better gradient flow -> easier optimization -> deeper networks become trainable


5. Why can ResNet be very deep ?

Because:
  - carries information directly forward
  - carries gradients directly backward
  -  adds no parameters when its an identity shortcut
  - lets each block learn smaller corrections rather than everything from scratch.

so;
>  ResNet = ordinary Network + shortcuts


6. Rules:
We cant add two tensors unless their dimensions match.

$F(x) + x$

must have the same shape.

if dimensions dont match, ResNet can use:

- Zero padding: add extra channels filled with zeros, no learnable patterns
- $1*1$ Convolution /projection : Use this convolution to change the number of channels or dimensions, has learnable patterns

7. ResNet 50 and beyond
For deeper models such as ResNet-50, ResNet-152, ResNet uses a bottleneck block to reduce computation and parameters . Instead of doing expensive convolution at a large channel size, it roughly does:
> 1×1 → 3×3 → 1×1     (compress → process → expand)


```
Normal network:
> y=F(x)

ResNet:
y=F(x)+x

because, the shortcut improves information and gradient flow, making very deep networks easier to train.

And this is the key connection to ConvNeXt: ConvNeXt is essentially a modernized ResNet, but with many design choices borrowed from Vision Transformers.
```



**ResNet Code**

Deep Residual Networks Pre Trained on ImageNet

All pre-trained models expect input images normalized similarly, i.e minibatches of 3 channel RGB image (3*H*W), where H and W are expected to be at least 224.

Because the pretrained model was trained with images prepared this way. If you give it differently scaled/distributed inputs, its learned weights won't work as effectively.

> Image → RGB → 224×224+ → [0,1] → normalize with ImageNet mean/std → model

In [ ]:
from transformers import ResNetForImageClassification

model=ResNetForImageClassification.from_pretrained("microsoft/resnet-50")
model.eval()

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

ResNetForImageClassification(
  (resnet): ResNetModel(
    (embedder): ResNetEmbeddings(
      (embedder): ResNetConvLayer(
        (convolution): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
        (normalization): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activation): ReLU()
      )
      (pooler): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
    (encoder): ResNetEncoder(
      (stages): ModuleList(
        (0): ResNetStage(
          (layers): Sequential(
            (0): ResNetBottleNeckLayer(
              (shortcut): ResNetShortCut(
                (convolution): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
                (normalization): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              )
              (layer): Sequential(
                (0): ResNetConvLayer(
                  (convolution): Conv2d(64

In [ ]:
# Sample Execution

from transformers import AutoImageProcessor, ResNetForImageClassification
import torch
from datasets import load_dataset

dataset=load_dataset("huggingface/cats-image")
image=dataset["test"]["image"][0]
image

feature_extractor=AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model=ResNetForImageClassification.from_pretrained("microsoft/resnet-50")

inputs=feature_extractor(image,return_tensors="pt")

with torch.no_grad():
  logits=model(**inputs).logits


predicted_label=logits.argmax(-1).item()
print(model.config.id2label[predicted_label])

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

tiger cat
